In [ ]:
!pip install -q google-genai transformers torch numpy qdrant-client underthesea uuid6
!pip install -q qdrant-client sentence-transformers pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.4 MB/s eta 0:00:00


### import and config

In [ ]:

from __future__ import annotations

import json

import re
import time
from pathlib import Path

import uuid6
from dataclasses import dataclass
from google import genai
from google.genai import types as _gtypes
from pydantic import BaseModel
import torch
from transformers import AutoModel, AutoTokenizer
from underthesea import word_tokenize
from google.colab import userdata
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct

# 60-cycle sexagenary calendar names (Giáp Tý → Quý Hợi)
CAN_CHI_60: list[str] = [
    "Giáp Tý",  "Ất Sửu",  "Bính Dần", "Đinh Mão", "Mậu Thìn", "Kỷ Tỵ",
    "Canh Ngọ", "Tân Mùi", "Nhâm Thân","Quý Dậu",  "Giáp Tuất","Ất Hợi",
    "Bính Tý",  "Đinh Sửu","Mậu Dần",  "Kỷ Mão",   "Canh Thìn","Tân Tỵ",
    "Nhâm Ngọ", "Quý Mùi", "Giáp Thân","Ất Dậu",   "Bính Tuất","Đinh Hợi",
    "Mậu Tý",   "Kỷ Sửu",  "Canh Dần", "Tân Mão",  "Nhâm Thìn","Quý Tỵ",
    "Giáp Ngọ", "Ất Mùi",  "Bính Thân","Đinh Dậu", "Mậu Tuất", "Kỷ Hợi",
    "Canh Tý",  "Tân Sửu", "Nhâm Dần", "Quý Mão",  "Giáp Thìn","Ất Tỵ",
    "Bính Ngọ", "Đinh Mùi","Mậu Thân", "Kỷ Dậu",   "Canh Tuất","Tân Hợi",
    "Nhâm Tý",  "Quý Sửu", "Giáp Dần", "Ất Mão",   "Bính Thìn","Đinh Tỵ",
    "Mậu Ngọ",  "Kỷ Mùi",  "Canh Thân","Tân Dậu",  "Nhâm Tuất","Quý Hợi",
]


# Sort longest-first to prevent sub-string matches (e.g. "Giáp" before "Giáp Tý")
_CAN_CHI_PATTERN: str = "|".join(
    re.escape(cc) for cc in sorted(CAN_CHI_60, key=len, reverse=True)
)

# Danh sách tên các tháng trong năm (Tháng 1 → Tháng 12)
MONTHS_LIST: list[str] = [
    "Tháng 1", "Tháng 2", "Tháng 3", "Tháng 4", "Tháng 5", "Tháng 6",
    "Tháng 7", "Tháng 8", "Tháng 9", "Tháng 10", "Tháng 11", "Tháng 12"
]

# Tạo Pattern mẫu dạng: (Tháng 1|Tháng 2|...|Tháng 12)
_MONTHS_PATTERN: str = "|".join(MONTHS_LIST)


# Regex hỗ trợ cả: [Mã]? + [Năm]? + [Can Chi hoặc Tháng] + [Nhìn trước]
CAN_CHI_MONTH_RE: re.Pattern = re.compile(
    r"^(?:\[\d{1,2}[ab]\]\s+)?(?:Năm\s+)?(?:" + _CAN_CHI_PATTERN + r"|" + _MONTHS_PATTERN + r")(?=\s*|\,|[\[\(\.]|$)",
    re.MULTILINE
)

# Tìm thẻ và trang trong văn bản [1], [2], …  , [PAGE:12]
FOOTNOTE_REF_RE: re.Pattern = re.compile(r"\[(\d{1,2})\]")
PAGE_MARKER_RE: re.Pattern = re.compile(r"\n\s*\[\[\[PAGE:(\d+)\]\]\]\s*\n")


# Separator priority list for word-aware text splitting
SEPARATORS: list[str] = ["\n\n", "\n", ".", "?", "!", ";", ":", ",", " ", ""]

#  cấu hình gemini
GEMINI_MODEL_ID: str = "gemini-3.1-flash-lite"
GEMINI_API_KEY: str = userdata.get("GEMINI_API_KEY")

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)


# cấu hình chunking
MODEL_KEY = "aiteamvn"

CHUNK_CONTEXT_LENGTH = 256

EMBED_MODEL_ID = "AITeamVN/Vietnamese_Embedding"

CHUNK_CONFIG = {
    # semantic target
    "ideal_min_chunks": 2,
    "ideal_max_chunks": 3,
    # hard cap by mini-chunk count
    "max_group_size": 5,
    # token constraints
    "target_token_min": 90,
    "target_token_max": 210,
    # hard safety limits
    "min_chunk_tokens": 70,
    "max_chunk_tokens": 240,
}

# Mini-chunk & merge sizing
MINI_CHUNK_WORDS: int  = 35     # ≈ 55-70 BPE tokens for classical Vietnamese
OVERLAP_TOK_MIN:  int  = 20
OVERLAP_TOK_MAX:  int  = 20

MIN_CHUNK_TOKENS = CHUNK_CONFIG["min_chunk_tokens"]
MAX_EMBED_TOKENS = CHUNK_CONFIG["max_chunk_tokens"]


# cấu hình qdrant
QDRANT_URL = userdata.get('QDRANT_URL')
API_KEY = userdata.get('QDRANT_API_KEY')

qdrant_client = QdrantClient(url=QDRANT_URL, api_key=API_KEY)

device = "cuda" if torch.cuda.is_available() else "cpu"

COLLECTION_NAME = "dvsktt_aiteamvn_chunks_256"

tokenizer = AutoTokenizer.from_pretrained(
    EMBED_MODEL_ID,
    trust_remote_code=True
)

embed_model = AutoModel.from_pretrained(
    EMBED_MODEL_ID,
    trust_remote_code=True
).to(device).eval()

VECTOR_SIZE = embed_model.config.hidden_size
print("MODEL:", MODEL_KEY)
print("MODEL ID:", EMBED_MODEL_ID)
print("COLLECTION:", COLLECTION_NAME)
print("VECTOR SIZE:", VECTOR_SIZE)

# Check if collection exists and its configuration matches
if qdrant_client .collection_exists(collection_name=COLLECTION_NAME):
    collection_info = qdrant_client .get_collection(collection_name=COLLECTION_NAME).config
    current_vector_size = collection_info.params.vectors.size

    if current_vector_size != VECTOR_SIZE:
        raise ValueError(
            f"Collection '{COLLECTION_NAME}' có vector size {current_vector_size}, "
            f"nhưng model hiện tại cần {VECTOR_SIZE}."
        )

    print(f"Collection '{COLLECTION_NAME}' đã tồn tại và khớp vector size {VECTOR_SIZE}.")
else:
    qdrant_client .create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,
            distance=Distance.COSINE
        )
    )
    print(f"Đã tạo collection mới: {COLLECTION_NAME}, vector size={VECTOR_SIZE}.")


### Data Structures
$$\text{Văn bản thô (chứa [[[PAGE:N]]])} \longrightarrow \text{PageSpan} \longrightarrow \text{MiniChunk} \longrightarrow \text{FinalChunk (Lưu Database)}$$

In [ ]:
@dataclass
class PageSpan:
    """One [PAGE:N] block from source text, with the marker stripped."""
    page_no:      int   # -1 if text precedes the first marker
    start_offset: int   # char offset in the reconstructed marker-free text
    end_offset:   int
    text:         str


@dataclass
class MiniChunk:
    """Small ~35-word fragment before LLM semantic merging."""
    idx:           int
    text:          str
    start_offset:  int   # absolute offset in clean-text coordinate space
    end_offset:    int
    pages:         list[int]
    footnote_refs: list[str]   # ordered, deduplicated


@dataclass
class FinalChunk:
    """Production-ready chunk post-embedding."""
    chunk_id:       str
    book_name:      str
    pages:          list[int]
    raw_text:       str          # exact source text, never mutated
    segmented_text: str          # underthesea-segmented version (for embedding)
    footnote_refs:  list[str]
    footnotes:      dict[str, str]
    token_count:    int
    # vector:         list[float]


###  PAGE PARSER
Tách trang: Lưu vị trí đầu trang, cuối trang, nội dung trang  
Trả về văn bản sạch không có số trang  
Trả về số trang khi người dùng đưa cho một PageSpan  


In [ ]:

def parse_page_spans(full_text: str) -> list[PageSpan]:
    # tìm thẻ page
    markers = list(PAGE_MARKER_RE.finditer(full_text))

    spans:   list[PageSpan] = []
    cursor = 0   # con trỏ vị trí (vị trí tiếp theo của trang)

 # Nếu không thấy thì là trang -1 và toàn bộ văn bản đều ở 1 trang
    if not markers:
        spans.append(PageSpan(-1, 0, len(full_text), full_text))

        return spans

    # xử lí văn bản trước thẻ PAGE
    pre = full_text[: markers[0].start()]
    if pre:
        spans.append(PageSpan(-1, 0, len(pre), pre))
        cursor = len(pre)   # chiều dài của văn bản trước PAGE

# Vòng lặp xử lý từng trang (i số page tìm được còn m là đối tượng )
    for i, m in enumerate(markers):
        page_no   = int(m.group(1))    # lấy phần sau (chứa số trang )
        raw_start = m.end()
        raw_end   = markers[i + 1].start() if i + 1 < len(markers) else len(full_text)
        page_text = full_text[raw_start:raw_end]   # văn bản trong 1 trang

        span_start = cursor
        span_end   = cursor + len(page_text)
        spans.append(PageSpan(page_no, span_start, span_end, page_text))    # lưu các thông tin của trang
        cursor = span_end

    return spans



# Tìm xem đoạn từ ký tự [start] đến [end] đang nằm ở (những) trang sách số mấy? trả về [4, 5]
def map_pages_to_slice(
    spans: list[PageSpan], start: int, end: int
) -> list[int]:
    result: list[int] = []
    for s in spans:
        if s.end_offset   <= start:
            continue
        if s.start_offset >= end:
            break
        if s.page_no != -1:
            result.append(s.page_no)
    return sorted(set(result))


### Cắt theo can chi (window text)
Cắt theo trang mặc định sau đó mới duyệt ngược lại cắt theo can chi

In [ ]:
@dataclass
class WindowResult:
    text: str
    pages: list[int]
    global_start: int
    global_end: int



def create_dynamic_can_chi_windows(
    spans: list[PageSpan], book_name: str
) -> list[WindowResult]:

    N = 3
    print(f"\n[WINDOWS] book='{book_name}' | window_size={N} pages")

    windows: list[WindowResult] = []
    remainder_text: str = ""
    remainder_origin: int = 0

    real_spans = [s for s in spans if s.page_no != -1]
    if not real_spans:
        return []

    idx = 0
    total = len(spans)

    while idx < total or remainder_text.strip():

        batch = spans[idx: idx + N]
        is_last = idx + len(batch) >= total

        if not batch and not remainder_text.strip():
            break

        batch_text = "".join(s.text for s in batch) if batch else ""

        if remainder_text:
            pool_text = remainder_text + batch_text
            pool_start = remainder_origin
        else:
            pool_text = batch_text
            pool_start = batch[0].start_offset if batch else remainder_origin

        if not pool_text.strip():
            idx += len(batch)
            continue

        if is_last:
            cut_local = len(pool_text)

        else:
            last_page_text = batch[-1].text if batch else ""
            last_page_start_in_pool = len(pool_text) - len(last_page_text)

            cut_local = -1

            matches = list(CAN_CHI_MONTH_RE.finditer(last_page_text))
            if matches:
                cut_local = last_page_start_in_pool + matches[-1].start()

            if cut_local <= 0:
                para = last_page_text.rfind("\n\n")
                if para != -1:
                    cut_local = last_page_start_in_pool + para + 2

            if cut_local <= 0:
                nl = last_page_text.rfind("\n")
                if nl != -1:
                    cut_local = last_page_start_in_pool + nl + 1

            if cut_local <= 0:
                dot = last_page_text.rfind(".")
                if dot != -1:
                    cut_local = last_page_start_in_pool + dot + 1

            if cut_local <= 0:
                cut_local = len(pool_text)

        window_text = pool_text[:cut_local]
        window_len = len(window_text)

        if (
            not is_last
            and window_len < 6000
            and (idx + len(batch)) < total
        ):
            extra_batch = spans[idx + len(batch): idx + len(batch) + 1]

            if extra_batch:
                extra_text = "".join(s.text for s in extra_batch)

                pool_text += extra_text

                # Tính lại điểm cắt dựa trên trang cuối mới
                last_page_text = extra_batch[-1].text
                last_page_start_in_pool = len(pool_text) - len(last_page_text)

                cut_local = -1

                matches = list(CAN_CHI_MONTH_RE.finditer(last_page_text))

                if matches:
                    cut_local = last_page_start_in_pool + matches[-1].start()

                if cut_local <= 0:
                    para = last_page_text.rfind("\n\n")
                    if para != -1:
                        cut_local = last_page_start_in_pool + para + 2

                if cut_local <= 0:
                    nl = last_page_text.rfind("\n")
                    if nl != -1:
                        cut_local = last_page_start_in_pool + nl + 1

                if cut_local <= 0:
                    dot = last_page_text.rfind(".")
                    if dot != -1:
                        cut_local = last_page_start_in_pool + dot + 1

                if cut_local <= 0:
                    cut_local = len(pool_text)

                window_text = pool_text[:cut_local]
                window_len = len(window_text)

                batch.extend(extra_batch)



        window_start = pool_start
        window_end = pool_start + cut_local

        remainder_text = pool_text[cut_local:]
        remainder_origin = pool_start + cut_local

        idx += len(batch)

        if not window_text.strip():
            if is_last:
                break
            continue

        pages = map_pages_to_slice(spans, window_start, window_end)

        windows.append(WindowResult(
            text=window_text,
            pages=pages,
            global_start=window_start,
            global_end=window_end,
        ))

        if is_last:
            break

    return windows

### Cấu trúc cắt
Cấu hình các thành phần có thể cắt tùy vào từng loại sách  

Cắt một chuỗi văn bản dài thành các đoạn nhỏ hơn


In [ ]:


def detect_structural_boundaries(text: str, book_name: str) -> list[int]:
    offsets: set[int] = {0, len(text)}

    for m in CAN_CHI_MONTH_RE.finditer(text):
            offsets.add(m.start())
    result = sorted(offsets)
    return result



def split_text_by_words(
    text: str, max_words: int, separators: list[str]
) -> list[tuple[int, int, str]]:
    if not text.strip():
        return []
    text = re.sub(r'(?<=[a-zà-ỹ])[\n\;](?=[a-zà-ỹ])', ' ', text)
    if len(text.split()) <= max_words:
        return [(0, len(text), text)]

    results: list[tuple[int, int, str]] = []
    cursor = 0

    while cursor < len(text):
        remaining = text[cursor:]
        if not remaining.strip():
            break

        if len(remaining.split()) <= max_words:
            results.append((cursor, len(text), remaining))
            break

        word_count = 0
        in_word    = False
        approx_end = len(remaining)
# đếm số lượng từ
        for ci, ch in enumerate(remaining):
            is_ws = ch in (" ", "\n", "\t", "\r")
            if is_ws:
                in_word = False
            elif not in_word:
                in_word = True
                word_count += 1
                if word_count > max_words:
                    approx_end = ci
                    break
        # print(f"Số lượng từ: {in_word} \n\n\n")
        # ── Walk backward to find the best separator at or before approx_end ─
        cut_local: int = -1
        for sep in separators:
            if sep == "":
                # Empty string = hard-cut at approx_end (last resort)
                cut_local = approx_end
                break
            search_bound = min(approx_end + len(sep), len(remaining))
            idx = remaining.rfind(sep, 0, search_bound)
            if idx != -1 and (idx + len(sep)) > 0:
                cut_local = idx + len(sep)
                break

        if cut_local <= 0:
            cut_local = approx_end
        if cut_local <= 0 or cut_local > len(remaining):
            cut_local = len(remaining)

        frag = remaining[:cut_local]   # exact source slice — no mutation
        if frag.strip():
            results.append((cursor, cursor + cut_local, frag))

        cursor += cut_local

    return results

### Mini Chunk Creation (rule-based pre-segmentation)
Từ các trang sách tách tiếp thành các thành phần bỏ hơn  
Từ các thành phần trên tách tiếp thành các chunk có kích thước khoản 35 word

In [ ]:


def create_mini_chunks(
    window: WindowResult, spans: list[PageSpan], book_name: str
) -> list[MiniChunk]:

    text       = window.text
    boundaries = detect_structural_boundaries(text, book_name)

    mini_chunks: list[MiniChunk] = []
    idx = 0

    for bi in range(len(boundaries) - 1):
        seg_start = boundaries[bi]
        seg_end   = boundaries[bi + 1]
        seg_text  = text[seg_start:seg_end]   # exact source slice
        if not seg_text.strip():
            continue

        frags = split_text_by_words(seg_text, MINI_CHUNK_WORDS, SEPARATORS)

        for (local_s, local_e, frag_text) in frags:
            if not frag_text.strip():
                continue

            # Absolute offsets in the global clean-text coordinate space
            abs_start = window.global_start + seg_start + local_s
            abs_end   = window.global_start + seg_start + local_e

            # Page mapping
            pages = map_pages_to_slice(spans, abs_start, abs_end)
            if not pages:
                pages = list(window.pages)   # fallback: window-level pages

            refs = list(dict.fromkeys(FOOTNOTE_REF_RE.findall(frag_text)))
            mini_chunks.append(MiniChunk(
                idx=idx,
                text=frag_text,
                start_offset=abs_start,
                end_offset=abs_end,
                pages=pages,
                footnote_refs=refs,
            ))
            idx += 1

    print(f"    [MINI-CHUNK] window chars={len(text):,} → {len(mini_chunks)} mini-chunks")
    return mini_chunks

### call llm



#### promt và gán nhãn cho input,  output

Luồng chạy: tạo form cho các minichunk sau đó sử dụng llm. llm sử dungj

In [ ]:
class SplitResponse(BaseModel):
    """
    Structured-output schema enforced on every LLM call (batch and sync).
    Guarantees the model can ONLY return a valid JSON object of this shape:
        {"split_after_indices": [3, 7, 12]}
    """
    split_after_indices: list[int]

def build_system_prompt(context_length: int, config: dict) -> str:
    return f"""\
You are an expert semantic chunk-boundary detector specializing in Vietnamese historical chronicles, royal annals, classical Vietnamese historical texts, and academic history documents.

YOUR TASK:
Analyze a sequence of pre-split Vietnamese historical text fragments called "mini-chunks".
Each mini-chunk is formatted as: <start_chunk_i tokens=X>text<end_chunk_i>
Your job is to determine ONLY the indices AFTER which a split must occur, so that the final macro-chunks are perfectly sized for a {context_length}-token embedding model.
STRICT OUTPUT RULES:
1. NEVER rewrite, summarize, translate, paraphrase, normalize, or correct the source text.
2. NEVER output any words, phrases, sentences, names, or quotes from the source text.
3. NEVER explain your reasoning.
4. NEVER use markdown fences.
5. OUTPUT ONLY one valid JSON object on a single line.
6. The JSON object must have exactly this shape:
{{"split_after_indices":[3,7,12]}}

INDEXING RULES:
- Indices are 0-based local mini-chunk indices.
- A split index i means: split AFTER mini-chunk i.
- Never return the final mini-chunk index.
- If no split is needed, return:
{{"split_after_indices":[]}}

VIETNAMESE HISTORICAL SEMANTIC SPLIT CRITERIA:
Insert a split when there is a clear shift in historical context, especially:
- A change in dynasty, reign era, historical year, or Can-Chi cycle.
- A transition to a new king, lord, general, envoy, scholar, official, or political actor.
- A shift to a different geographic location, battlefield, province, diplomatic mission, or administrative area.
- A transition between different event types, such as war, diplomacy, succession, taxation, law, ritual, disaster, rebellion, punishment, appointment, or royal order.
- A structural transition between narrative text, royal edict, decree, memorial, historian commentary, critique, citation, or explanatory note.
- A clear ending of one historical event before the text moves to another event.

ANTI-OVERFRAGMENTATION RULE:
Do NOT split merely because of sentence boundaries.
Do NOT split if the mini-chunks are part of the same continuous event, battle, royal discussion, decree, dialogue, campaign, or explanation.
Keep related historical context together when it remains coherent and within the token limits.
HARD RULE: NEVER split if the current macro-chunk accumulated so far contains fewer than {config["ideal_min_chunks"]} mini-chunks (or is under {config["target_token_min"]} token), unless absolutely forced by a shift in Dynasty. Prioritize grouping smaller events together into a single macro-chunk.
TOKEN SIZE TARGETS:
The embedding model context length is {context_length} tokens.
- Every mini-chunk now has a 'tokens' attribute (e.g., tokens=55).
- HARD LIMIT: Never allow the accumulated tokens of any macro-chunk to exceed {config["max_chunk_tokens"]} tokens.
- If a continuous historical topic is too long and the next chunk would push the total sum over {config["max_chunk_tokens"]} tokens, you MUST force a split immediately at the current index, even if the semantic event is not finished!
The preferred final macro-chunk range is approximately {config["target_token_min"]} to {config["target_token_max"]} tokens.
Avoid creating chunks below {config["min_chunk_tokens"]} tokens unless the text is a complete standalone historical unit.
Avoid creating chunks above {config["max_chunk_tokens"]} tokens.

MINI-CHUNK GROUPING TARGETS:
Prefer {config["ideal_min_chunks"]} to {config["ideal_max_chunks"]} mini-chunks per macro-chunk when the historical context is coherent.
No macro-chunk may contain more than {config["max_group_size"]} mini-chunks.

FORCED SPLIT RULE:
If no clear semantic boundary appears, force a split before the macro-chunk would exceed:
- {config["max_group_size"]} mini-chunks, or
- approximately {config["max_chunk_tokens"]} tokens.

PRIORITY ORDER:
1. Preserve historical semantic coherence.
2. Avoid exceeding token and mini-chunk limits.
3. Avoid very small chunks.
4. Prefer natural historical boundaries over fixed-size boundaries.

OUTPUT FORMAT:
Return EXACTLY one JSON object:
{{"split_after_indices":[...]}}
"""

_USER_TEMPLATE: str = """\
The following {n} tagged mini-chunks are a sequential sub-batch from a Vietnamese historical text.
Review the text content and indices carefully. Output the split indices according to the system instructions.

[START OF MINI-CHUNKS]
{tagged}
[END OF MINI-CHUNKS]

Your JSON response:"""


# form cho các minichunk trước khi gửi cho llm
def _format_tagged_chunks(chunks: list[MiniChunk]) -> str:
    return "\n".join(
        f"<start_chunk_{mc.idx}>{mc.text}<end_chunk_{mc.idx}>"
        for mc in chunks
    )


# Nhận câu trả lời của llm trả về mảng int chứa các vị trí cần cắt
def _parse_split_response(raw: str, n_chunks: int, max_group_size: int,) -> list[int]:
    clean = re.sub(r"^```[a-z]*\n?", "", raw.strip())
    clean = re.sub(r"\n?```$", "", clean.strip())
    clean = re.sub(r"<think>.*?</think>", "", clean, flags=re.DOTALL).strip()

    parsed = json.loads(clean)
    indices = parsed.get("split_after_indices", [])

    max_valid = n_chunks - 2

    #  Lọc split từ LLM
    splits = sorted(
        set(
            int(i) for i in indices
            if isinstance(i, (int, float)) and 0 <= int(i) <= max_valid
        )
    )

    #  Ép cứng: không group nào được dài hơn max_group_size
    forced_splits = []
    group_start = 0

    for split_idx in splits:
        while split_idx - group_start + 1 > max_group_size:
            forced_idx = group_start + int(max_group_size / 2)
            if forced_idx <= max_valid:
                forced_splits.append(forced_idx)
            group_start = forced_idx + int(max_group_size / 2)

        forced_splits.append(split_idx)
        group_start = split_idx + 1

    #  Xử lý đoạn cuối sau split cuối cùng
    last_idx = n_chunks - 1
    while last_idx - group_start + 1 > max_group_size:
        forced_idx = group_start + max_group_size - 1
        if forced_idx <= max_valid:
            forced_splits.append(forced_idx)
        group_start = forced_idx + 1

    return sorted(set(forced_splits))


#### call llm

In [ ]:

def call_gemini(system_prompt: str, user_prompt: str) -> str:
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL_ID,
        contents=user_prompt,
        config=_gtypes.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=0.0,
            max_output_tokens=4096,
            response_mime_type="application/json",
        ),
    )

    if not response.text:
        raise RuntimeError("Gemini không trả về nội dung.")

    return response.text


####  Gọi LLM để gộp các minichunk theo ngữ nghĩa

In [ ]:
def llm_semantic_split(
    mini_chunks: list[MiniChunk],
    context_length: int,
    chunk_config: dict,
    max_retries: int = 3,
) -> list[int]:

    if len(mini_chunks) < 3:
        print(f"    [GEMINI] Bỏ qua vì chỉ có " f"{len(mini_chunks)} mini-chunk.")
        return []

    tagged_chunks = _format_tagged_chunks(mini_chunks)

    user_message = _USER_TEMPLATE.format(
        n=len(mini_chunks),
        tagged=tagged_chunks,
    )

    system_prompt = build_system_prompt(context_length, chunk_config)

    number_of_chunks = len(mini_chunks)
    max_group_size = chunk_config["max_group_size"]

    for attempt in range(1, max_retries + 1):
        raw_response = ""

        try:
            print(
                f"    [GEMINI] Calling {GEMINI_MODEL_ID} "
                f"(attempt {attempt}/{max_retries})..."
            )

            raw_response = call_gemini(system_prompt, user_message)

            print(f"\n[GEMINI RAW RESPONSE]\n{raw_response}\n")

            valid_splits = _parse_split_response(
                raw=raw_response,
                n_chunks=number_of_chunks,
                max_group_size=max_group_size,
            )

            print(
                f"    [GEMINI] chunks={number_of_chunks} "
                f"→ splits={valid_splits}"
            )

            return valid_splits

        except json.JSONDecodeError as exc:
            print(
                f"    [GEMINI] JSON error "
                f"(attempt {attempt}/{max_retries}): {exc}"
            )
            print(
                f"    [GEMINI] Raw response: "
                f"{repr(raw_response[:200])}"
            )

        except Exception as exc:
            print(
                f"    [GEMINI] Error "
                f"(attempt {attempt}/{max_retries}): "
                f"{type(exc).__name__}: {exc}"
            )

        if attempt < max_retries:
            wait_seconds = 2 ** attempt

            print(
                f"    [GEMINI] Waiting "
                f"{wait_seconds} seconds before retry..."
            )

            time.sleep(wait_seconds)

    print(
        "    [GEMINI] Tất cả lần gọi đều thất bại. "
        "Window này sẽ không có semantic split."
    )

    return []

### Semantic Merging
Merge theo kết quả của gemini trả về  
Và overlap các chunk

In [ ]:


def _tok_count_full(text: str, tokenizer) -> int:
    return len(tokenizer.encode(text, add_special_tokens=True, truncation=False))

 # group theo danh sách mà llm gợi ý
def _group_by_splits(
    chunks: list[MiniChunk], split_indices: list[int]
) -> list[list[MiniChunk]]:

    if not chunks:
        return []
    split_set = set(split_indices)
    groups:  list[list[MiniChunk]] = []
    current: list[MiniChunk]       = []
    # thêm các chunks vào current. Nếu gặp điểm cắt thì đẩy vào groups rồi làm sạch
    for mc in chunks:
        current.append(mc)
        if mc.idx in split_set:
            groups.append(current)
            current = []
    if current:
        groups.append(current)
    return groups


  #  raw_text   — văn bản gốc
  # embed_input— raw_text + a semantic overlap slice
  # groups chứa list chunks. trong chunks có các minichunks
def build_chunk_texts(
    groups: list[list[MiniChunk]], tokenizer
) -> list[tuple[str, str]]:

    results: list[tuple[str, str]] = []

    for i, group in enumerate(groups):
        raw   = "".join(mc.text for mc in group)
        embed = raw   # default: no overlap

        if i + 1 < len(groups): # lấy văn bản của chunk tiếp theo
            next_raw  = "".join(mc.text for mc in groups[i + 1])
            next_toks = tokenizer.encode(next_raw, add_special_tokens=False,
                                         truncation=False)
            n_next = len(next_toks)
            MAX = OVERLAP_TOK_MAX
            MIN = OVERLAP_TOK_MIN
            if(n_next <=  CHUNK_CONFIG["min_chunk_tokens"]):
                MAX = MAX * 3
                MIN = MIN * 3
            elif(n_next <=  (CHUNK_CONFIG["min_chunk_tokens"] + MAX)):
                MAX = MAX * 2
                MIN = MIN * 2

            if n_next > 0:
                for ovl in range(MAX, MIN - 1, -1):
                    # Estimate character count corresponding to `ovl` tokens
                    # using the token/char ratio of next_raw.
                    ratio    = min(ovl / n_next, 1.0)
                    char_end = max(1, int(len(next_raw) * ratio))

                    # Snap forward to the next whitespace to avoid mid-word cut
                    while (char_end < len(next_raw) and
                           next_raw[char_end] not in (" ", "\n", "\t", "\r")):
                        char_end += 1

                    ovl_slice = next_raw[:char_end]   # pure source slice
                    candidate = raw + " " + ovl_slice

                    if _tok_count_full(candidate, tokenizer) <= CHUNK_CONTEXT_LENGTH:
                        embed = candidate
                        break
                    # If MIN overlap still overflows, leave embed = raw
        embed = re.sub((r"\[(\d{1,2})\]"), ' ', embed)
        results.append((raw, embed))

    return results


 # gắn footer, pape, header
def collect_group_metadata(
    groups: list[list[MiniChunk]],
    page_footnotes_map: dict,
    page_headers_map: dict
) -> list[dict]:
    metas: list[dict] = []

    for group in groups:
        all_pages: list[int] = []
        all_refs: list[str] = []

        for mc in group:
            all_pages.extend(mc.pages)
            all_refs.extend(mc.footnote_refs)

        pages = sorted(set(all_pages))
        refs = list(dict.fromkeys(all_refs))  # ordered dedup

        # Lấy header theo từng trang
        page_headers: dict[str, str] = {}

        for pg in pages:
            pg_str = str(pg)

            if pg_str in page_headers_map:
                page_headers[pg_str] = page_headers_map[pg_str]

        # Lấy footnote theo từng trang để tránh lỗi trùng ref [1], [2] giữa nhiều trang
        footnotes_by_page: dict[str, dict[str, str]] = {}

        for pg in pages:
            pg_str = str(pg)

            if pg_str not in page_footnotes_map:
                continue

            for ref in refs:
                if ref in page_footnotes_map[pg_str]:
                    footnotes_by_page.setdefault(pg_str, {})[ref] = page_footnotes_map[pg_str][ref]

        metas.append({
            "pages": pages,
            "page_headers": page_headers,
            "footnote_refs": refs,
            "footnotes": footnotes_by_page,
        })

    return metas



### Word Segmentation (underthesea) and tokenizer

In [ ]:


def segment_underthesea(text: str) -> str:
    return word_tokenize(text, format="text")


def inspect_tokens(
    raw: str, seg: str, tokenizer, chunk_id: str
) -> int:
    ids   = tokenizer.encode(
        seg,
        add_special_tokens=True,
        truncation=True,
        max_length=CHUNK_CONTEXT_LENGTH,
    )
    count = len(ids)
    # print(count)

    print(f"      [TOKEN] id={chunk_id[:8]}… | "
          f"raw[0:60]={raw[:60].strip()!r}")
    print(f"              seg[0:60]={seg[:60].strip()!r}")
    print(f"              ids[:8]={ids[:8]} | total_tokens={count}")
    return count



### Embedding Model

In [ ]:

def _mean_pool(model_output, attention_mask: torch.Tensor) -> torch.Tensor:
    """Compute attention-mask-weighted mean pool over token embeddings."""
    tok_emb  = model_output.last_hidden_state            # (B, T, D)
    mask_exp = attention_mask.unsqueeze(-1).expand(tok_emb.size()).float()
    return torch.sum(tok_emb * mask_exp, dim=1) / mask_exp.sum(dim=1).clamp(min=1e-9)


def embed_text(
    seg_text: str,
    tokenizer,
    model:  AutoModel,
    device: str,
) -> list[float]:
    enc = tokenizer(
        seg_text,
        padding=True,
        truncation=True,
        max_length=CHUNK_CONTEXT_LENGTH,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        out = model(**enc)

    emb = _mean_pool(out, enc["attention_mask"])
    emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    return emb[0].cpu().numpy().tolist()


###  QDRANT PAYLOAD

$$\text{Chuỗi text thô} \xrightarrow{\text{Underthesea}} \text{word segmented} \xrightarrow{\text{Encoder + Pooling}} \text{Vector [768 chiều]} \xrightarrow{\text{Đóng gói}} \text{Qdrant Database}$$

In [ ]:

def to_qdrant_point(chunk: FinalChunk) -> dict:
    return {
        "id":      chunk.chunk_id,
        "payload": {
            "book_name":      chunk.book_name,
            "pages":          chunk.pages,
            "raw_text":       chunk.raw_text,
            "segmented_text": chunk.segmented_text,
            "footnotes":      chunk.footnotes,
            "token_count":    chunk.token_count,
        },
    }



### MAIN
```
run_pipeline(input_json, backends)

1. Lấy tên sách, text, footnote
2. Tách full_text thành page spans
3. Gom page spans thành windows
4. Tạo mini-chunks cho từng window
5. Nếu dùng batch:
      gửi tất cả mini-chunk cho Gemini Batch để lấy split indices
   Nếu không:
      lát nữa gọi LLM từng window
6. Với từng window:
      ->  lấy mini-chunk
      ->  lấy split indices từ batch hoặc gọi LLM sync
      ->  gộp mini-chunk thành semantic groups
      ->  tạo raw_text + embed_input
      ->  lấy metadata trang và footnote
      ->  segment tiếng Việt bằng underthesea
      ->  đếm token
7. đẩy points lên Qdrant cloud
8. Trả về danh sách point
```

In [ ]:
def run_pipeline(
    input_json:         dict,
) -> list[dict]:

    book_name          = input_json["book_name"]
    full_text          = input_json["full_text"]
    page_footnotes_map = input_json["page_footnotes_map"]
    page_headers_map = input_json["page_headers_map"]

    print("\n" + "═" * 72)
    print(f"  [PHASE 1] RUN CHUNKING PIPELINE — '{book_name}'")
    print(f"  Using Context Length Target: {CHUNK_CONTEXT_LENGTH}")
    print("═" * 72)
# bóc tách trang
    spans = parse_page_spans(full_text)

# chia sách thành các trang cho các lần gọi api
    windows = create_dynamic_can_chi_windows(spans, book_name)

    all_chunks:    list[dict] = []
    total_windows: int        = len(windows)


# chia từng minichunk cho danh sách trang trên
    for wi, window in enumerate(windows):

        pg_str = str(window.pages[:6]) + ("…" if len(window.pages) > 6 else "")
        print(f"  WINDOW {wi + 1:>4}/{total_windows} | pages={pg_str} | chars={len(window.text):,}")


        mini_chunks = create_mini_chunks(window, spans, book_name)
        if not mini_chunks:
            print("    [SKIP] No content mini-chunks produced for this window.")
            continue

        print(f"    [STEP5-SYNC] Calling LLM chain…")
        split_indices = llm_semantic_split( mini_chunks, context_length=CHUNK_CONTEXT_LENGTH, chunk_config=CHUNK_CONFIG,)
        print(f"llm trả về \n {split_indices} \n\n\n")

# Merge groups + overlap + footnote metadata
        groups       = _group_by_splits(mini_chunks, split_indices)
        chunk_texts  = build_chunk_texts(groups, tokenizer)    # (raw, embed_input)
        metas        = collect_group_metadata(groups, page_footnotes_map, page_headers_map)

        print(f"    [MERGE] {len(mini_chunks)} mini-chunks + {len(split_indices)} splits "
              f"→ {len(groups)} semantic groups")

        points_to_upload: list[PointStruct] = []
# Segment, inspect, embed, format
        for gi, ((raw_text, embed_input), meta) in enumerate(zip(chunk_texts, metas)):
            if not raw_text.strip():
                continue

            chunk_id = str(uuid6.uuid7())

      # Word segmentation (applied to embed_input which includes overlap)
            seg_text = segment_underthesea(embed_input)

      #  Token inspection + count
            tok_count = inspect_tokens(raw_text, embed_input, tokenizer, chunk_id)

            vector = embed_text(embed_input, tokenizer, embed_model, device)
            if hasattr(vector, "tolist"):
                vector = vector.tolist()

            chunk_payload = {
                "chunk_id": chunk_id,
                "book_name": book_name,
                "pages": meta["pages"],
                "raw_text": raw_text,
                "embed_input": embed_input,
                "segmented_text": seg_text,
                "headers": meta["page_headers"],
                "footnotes": meta["footnotes"],
                "token_count": tok_count
            }
            all_chunks.append(chunk_payload)

            point = PointStruct(
                id=chunk_id,
                vector=vector,
                payload=chunk_payload,
            )

            points_to_upload.append(point)

        if points_to_upload:
            qdrant_client.upsert(
                collection_name=COLLECTION_NAME,
                points=points_to_upload,
                wait=True,
            )

            print(f"[QDRANT] Uploaded {len(points_to_upload)} points.")
        else:
            print("[QDRANT] Không có point để upload.")

        # Tránh gọi Gemini quá liên tục
        if wi < total_windows:
            time.sleep(2.0)


        time.sleep(2.0)
    return all_chunks

#  run chunking Đại Việt Sử Ký

In [ ]:

JSON_PATH = "full_DVSK_data.json"           # Đại Việt Sử Ký Toàn Thư

OUTPUT_PATH = JSON_PATH.replace(".json", f"_{MODEL_KEY}_{CHUNK_CONTEXT_LENGTH}.json")

input_path = Path(JSON_PATH)

if not input_path.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file đầu vào: {JSON_PATH}"
    )

with input_path.open(mode="r", encoding="utf-8",) as file:
    input_data: dict = json.load(file)

chunks: list[dict] = run_pipeline(
    input_json=input_data,
)

print( f"\n[SAVE] Writing {len(chunks):,} chunks "f"to '{OUTPUT_PATH}'...")

output_path = Path(OUTPUT_PATH)

with output_path.open(mode="w", encoding="utf-8",) as file:
    json.dump(chunks, file, ensure_ascii=False, indent=2)

output_size_kb = output_path.stat().st_size / 1024

print( f"[SAVE] Done. " f"{output_size_kb:,.1f} KB written.")


Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
              ids[:8]=[0, 4428, 195, 16859, 86729, 1893, 62322, 38262] | total_tokens=136
      [TOKEN] id=019f17e9… | raw[0:60]='Hậu họ Nguyễn, tên huý là Đạo, con gái viên quản lĩnh ở hươ'
              seg[0:60]='Hậu họ Nguyễn, tên huý là Đạo, con gái viên quản lĩnh ở hươ'
              ids[:8]=[0, 200554, 7424, 19098, 4, 10587, 4571, 1673] | total_tokens=158
      [TOKEN] id=019f17e9… | raw[0:60]='Không bao lâu, làm tới Hình bộ thượng thư kiêm Bảo Thiên đi'
              seg[0:60]='Không bao lâu, làm tới Hình bộ thượng thư kiêm Bảo Thiên đi'
              ids[:8]=[0, 13887, 8609, 25825, 4, 1839, 7067, 79604] | total_tokens=107
      [TOKEN] id=019f17e9… | raw[0:60]='Linh Ẩn gian dâm với vợ lẽ của cha, để tang cha mẹ ít ngày,'
              seg[0:60]='Linh Ẩn gian dâm với vợ lẽ của cha, để tang cha mẹ ít ngày,'
              ids[:8]=[0, 55120, 6, 197330, 19, 6051, 162620, 1116] | total_tokens=223
      [TOKEN] id=019f17e9… | r

In [38]:
import json

with open("full_DVSK_data_aiteamvn_256.json", "r", encoding="utf-8") as f:
    data = json.load(f)

token_count = [
    item["token_count"]
    for item in data
    if "token_count" in item and isinstance(item["token_count"], (int, float))
]

avg_tok_count = sum(token_count) / len(token_count) if token_count else 0

print(f"Số lượng item có token_count: {len(token_count)}")
print(f"Trung bình token_count: {avg_tok_count:.2f}")

Số lượng item có token_count: 5343
Trung bình token_count: 143.05
